# Week 05 — Model / Analysis

This notebook builds a model that is compared with the Week-04 baseline on the **same rows and same holdout split**.

The notebook is deliberately conservative:
- it detects the local FlyRank dataset instead of inventing data;
- it excludes obvious label-derived and future-window columns from features;
- it uses a simple Logistic Regression baseline model first;
- it uses a stratified holdout for classification;
- it reports useful metrics and error examples;
- it stops with an explicit self-check if the required target or baseline cannot be identified.

**Important:** edit `TARGET_COLUMN` only if automatic target detection does not find the correct lane target.


In [ ]:
from pathlib import Path
import re
import json
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, confusion_matrix,
    classification_report
)
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent.parent

OUTPUT_DIR = ROOT / "work" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20

print("Repository root:", ROOT)


## 1) Method choice and why

**Chosen method: Logistic Regression for classification.**

The Week-4 baseline is a transparent hand-built scoring rule. A regularized Logistic Regression is an appropriate first model because:
- it is simple enough to audit;
- it provides class probabilities;
- coefficients can be interpreted;
- it is substantially less likely to hide data problems than immediately choosing a complex ensemble;
- it gives a meaningful benchmark for whether learned relationships improve on the hand-built baseline.

If the target is not binary/multiclass classification, this notebook stops rather than silently applying the wrong model.


In [ ]:
def norm(x):
    return re.sub(r"[^a-z0-9]+", "_", str(x).lower()).strip("_")

# Find candidate data.
files = []
for d in [ROOT/"data", ROOT/"work"/"data", ROOT/"flyrank", ROOT/"work"]:
    if d.exists():
        files.extend(d.rglob("*.csv"))
        files.extend(d.rglob("*.parquet"))

files = [
    p for p in files
    if "outputs" not in p.parts
    and "baseline_action_score" not in p.name
    and not p.name.startswith(".")
]

if not files:
    raise FileNotFoundError(
        "No source CSV/Parquet file found. Put the FlyRank dataset in data/ or work/data/."
    )

data_path = max(files, key=lambda p: p.stat().st_size)
df = pd.read_csv(data_path) if data_path.suffix.lower() == ".csv" else pd.read_parquet(data_path)

print("Selected data:", data_path.relative_to(ROOT))
print("Shape:", df.shape)
print("Columns:")
print(list(df.columns))


In [ ]:
# ---- Target configuration ----
# If automatic detection is wrong, replace None with the exact target column name.
TARGET_COLUMN = None

# Explicit exclusions can be added here if your data contract identifies columns
# that are post-outcome, IDs, or otherwise not valid model inputs.
EXCLUDE_COLUMNS = set()

label_keywords = [
    "label", "target", "outcome", "converted", "conversion",
    "ground_truth", "groundtruth", "is_quick_win", "quick_win"
]

if TARGET_COLUMN is None:
    candidates = [
        c for c in df.columns
        if any(k in norm(c) for k in label_keywords)
    ]
    # Prefer an exact target/label name.
    exact = [c for c in candidates if norm(c) in {"target", "label", "outcome"}]
    TARGET_COLUMN = exact[0] if exact else (candidates[0] if candidates else None)

print("Detected target:", TARGET_COLUMN)

if TARGET_COLUMN is None:
    raise ValueError(
        "No target column was detected. Set TARGET_COLUMN to the lane's actual target "
        "from the Week-2/Week-3 data contract."
    )

y = df[TARGET_COLUMN]
print("Target dtype:", y.dtype)
print("Target distribution:")
display(y.value_counts(dropna=False).to_frame("n"))

if y.nunique(dropna=True) < 2:
    raise ValueError("The target has fewer than two usable classes.")


## 2) Split design

A single stratified 80/20 holdout is used for classification.

The **same row split** will be used to evaluate:
1. the learned Logistic Regression model; and
2. the Week-4 baseline score/action.

This prevents an unfair comparison where one method gets a different evaluation population.

No future information is used to create features.


In [ ]:
# Clean target and create stable row IDs.
data = df.copy()
data = data.dropna(subset=[TARGET_COLUMN]).reset_index(drop=False).rename(columns={"index": "_original_row"})

y = data[TARGET_COLUMN]
X = data.drop(columns=[TARGET_COLUMN])

# Remove obvious identifiers and leakage-like columns.
automatic_exclude = set()
for c in X.columns:
    n = norm(c)
    if (
        n in {"id", "query_id", "keyword_id", "url_id", "page_id", "item_id"}
        or n.endswith("_id")
        or any(k in n for k in [
            "future", "next_period", "post_outcome", "outcome_date",
            "conversion_date", "label", "target", "ground_truth", "quick_win"
        ])
    ):
        automatic_exclude.add(c)

# Date columns are not directly used; if you need recency, create it from the
# training-period reference date in a data-contract-approved way.
for c in X.columns:
    if pd.api.types.is_datetime64_any_dtype(X[c]):
        automatic_exclude.add(c)

feature_exclude = automatic_exclude | EXCLUDE_COLUMNS
feature_cols = [c for c in X.columns if c not in feature_exclude]

X = X[feature_cols].copy()

# Drop all-null columns.
all_null = [c for c in X.columns if X[c].isna().all()]
X = X.drop(columns=all_null)
feature_cols = list(X.columns)

print("Excluded columns:", sorted(feature_exclude))
print("Feature count:", len(feature_cols))
print("Features:", feature_cols)

if len(feature_cols) == 0:
    raise ValueError("No usable features remain after leakage/ID exclusions.")

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, data["_original_row"],
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train class distribution:")
display(y_train.value_counts(normalize=True).rename("share").to_frame())
print("Test class distribution:")
display(y_test.value_counts(normalize=True).rename("share").to_frame())


## 3) Train + compare vs Week-4 baseline

The model uses preprocessing inside a scikit-learn Pipeline so that imputation, scaling, and one-hot encoding are fit **only on the training split**.

The baseline is loaded from `work/outputs/baseline_action_score.csv`. Its action label is treated as a prediction, not as the ground-truth target.

For a fair comparison, baseline rows are matched back to the original data using the row identifier where possible. If the Week-4 baseline used another identifier, update the mapping cell.


In [ ]:
# Feature preprocessing
numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = [c for c in X_train.columns if c not in numeric_features]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scale", StandardScaler())
        ]), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features)
    ],
    remainder="drop"
)

model = Pipeline([
    ("preprocess", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

model.fit(X_train, y_train)
pred = model.predict(X_test)
proba = model.predict_proba(X_test) if hasattr(model, "predict_proba") else None

print("Model trained.")


In [ ]:
# Metrics for the learned model.
labels = np.unique(y_test)

model_metrics = {
    "accuracy": accuracy_score(y_test, pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, pred),
    "precision_weighted": precision_score(y_test, pred, average="weighted", zero_division=0),
    "recall_weighted": recall_score(y_test, pred, average="weighted", zero_division=0),
    "f1_weighted": f1_score(y_test, pred, average="weighted", zero_division=0),
}

if len(labels) == 2 and proba is not None:
    model_metrics["roc_auc"] = roc_auc_score(y_test, proba[:, 1])

print("Model metrics:")
display(pd.DataFrame([model_metrics], index=["Logistic Regression"]).T.rename(columns={0:"value"}))

print("\nClassification report:")
print(classification_report(y_test, pred, zero_division=0))


In [ ]:
# ---- Load Week-4 baseline ----
baseline_path = OUTPUT_DIR / "baseline_action_score.csv"

if not baseline_path.exists():
    raise FileNotFoundError(
        "Week-4 baseline CSV is missing. Run work/notebooks/w04_baseline_score.ipynb first "
        "so it creates work/outputs/baseline_action_score.csv."
    )

baseline = pd.read_csv(baseline_path)
print("Baseline columns:", list(baseline.columns))
display(baseline.head())

if "action" not in baseline.columns:
    raise ValueError("The Week-4 baseline CSV must contain an 'action' column.")

# The generated Week-4 notebook uses _item_id based on the selected ID column or
# original row index. Match the baseline to this dataset.
if "_item_id" in baseline.columns:
    baseline_id = baseline["_item_id"].astype(str)
else:
    raise ValueError("Baseline CSV has no _item_id column; update this matching cell.")

# Recreate the Week-4 row identifier convention.
possible_id_cols = [
    c for c in data.columns
    if norm(c) in {"id","query_id","keyword_id","url_id","page_id","item_id"}
    or norm(c).endswith("_id")
]

if possible_id_cols:
    match_col = possible_id_cols[0]
    current_ids = data[match_col].astype(str)
    print("Matching baseline using:", match_col)
else:
    current_ids = data["_original_row"].astype(str)
    print("Matching baseline using original row index.")

baseline_map = pd.DataFrame({
    "_baseline_id": baseline["_item_id"].astype(str),
    "baseline_action": baseline["action"].astype(str)
})

eval_rows = pd.DataFrame({
    "_original_row": idx_test.values,
    "y_true": y_test.values,
    "model_pred": pred
})

eval_rows["_baseline_id"] = (
    data.loc[idx_test, possible_id_cols[0]].astype(str).values
    if possible_id_cols else idx_test.astype(str)
)

eval_rows = eval_rows.merge(
    baseline_map,
    left_on="_baseline_id",
    right_on="_baseline_id",
    how="left"
)

print("Baseline match rate:", eval_rows["baseline_action"].notna().mean())

if eval_rows["baseline_action"].isna().any():
    print("WARNING: Some test rows did not match the baseline. Those rows will be excluded from baseline metrics.")


In [ ]:
# Compare using the SAME matched test rows and the SAME ground-truth target.
baseline_eval = eval_rows.dropna(subset=["baseline_action"]).copy()

if len(baseline_eval) == 0:
    raise ValueError("No test rows matched the Week-4 baseline.")

baseline_pred = baseline_eval["baseline_action"]

# Baseline actions may not use the same labels as the target. Only compare directly
# if their label sets are compatible.
target_labels = set(pd.Series(baseline_eval["y_true"]).astype(str).unique())
baseline_labels = set(baseline_pred.astype(str).unique())

if target_labels != baseline_labels:
    print("Baseline action labels:", sorted(baseline_labels))
    print("Ground-truth target labels:", sorted(target_labels))
    raise ValueError(
        "The Week-4 action labels do not match the ground-truth target labels. "
        "Do not force this comparison. Map the baseline action to the actual target "
        "only if your data contract explicitly defines that mapping."
    )

baseline_metrics = {
    "accuracy": accuracy_score(baseline_eval["y_true"], baseline_pred),
    "balanced_accuracy": balanced_accuracy_score(baseline_eval["y_true"], baseline_pred),
    "precision_weighted": precision_score(baseline_eval["y_true"], baseline_pred, average="weighted", zero_division=0),
    "recall_weighted": recall_score(baseline_eval["y_true"], baseline_pred, average="weighted", zero_division=0),
    "f1_weighted": f1_score(baseline_eval["y_true"], baseline_pred, average="weighted", zero_division=0),
}

comparison = pd.DataFrame([
    model_metrics,
    baseline_metrics
], index=["Logistic Regression", "Week-4 Baseline"])

display(comparison)


### Interpretation

Do not choose the model simply because it is more complex.

The useful question is whether the model improves the chosen metric on the same evaluation rows. If the baseline wins, the correct conclusion is that the Week-4 rule is currently stronger or that the model/features are not yet sufficient.


In [ ]:
primary_metric = "balanced_accuracy"

if primary_metric not in comparison.columns:
    primary_metric = "f1_weighted"

model_value = comparison.loc["Logistic Regression", primary_metric]
baseline_value = comparison.loc["Week-4 Baseline", primary_metric]
delta = model_value - baseline_value

print(f"Primary metric: {primary_metric}")
print(f"Model:    {model_value:.4f}")
print(f"Baseline: {baseline_value:.4f}")
print(f"Delta:    {delta:+.4f}")

if delta > 0:
    print("Conclusion: the model beats the Week-4 baseline on the primary metric.")
elif delta < 0:
    print("Conclusion: the Week-4 baseline beats the model on the primary metric.")
else:
    print("Conclusion: the model and baseline are tied on the primary metric.")


## 4) Errors and interpretation

The error table shows false positives and false negatives where applicable. These examples should be read against the actual feature values and source records before making a causal claim.


In [ ]:
error_rows = eval_rows.copy()
error_rows["correct"] = error_rows["y_true"].astype(str) == error_rows["model_pred"].astype(str)
errors = error_rows[~error_rows["correct"]].copy()

print("Test rows:", len(eval_rows))
print("Model errors:", len(errors))
print("Error rate:", len(errors) / max(len(eval_rows), 1))

display(errors.head(10))

# Feature permutation importance on the held-out set.
try:
    perm = permutation_importance(
        model, X_test, y_test,
        scoring="balanced_accuracy",
        n_repeats=5,
        random_state=RANDOM_STATE
    )
    importance = pd.DataFrame({
        "feature": X_test.columns,
        "importance_mean": perm.importances_mean,
        "importance_std": perm.importances_std
    }).sort_values("importance_mean", ascending=False)

    print("Top permutation importances:")
    display(importance.head(15))
except Exception as e:
    print("Permutation importance unavailable:", repr(e))


In [ ]:
# Produce a short, data-grounded error interpretation.
if len(errors) == 0:
    error_summary = (
        "No errors occurred on the displayed holdout sample. This should not be treated as proof "
        "that the model is perfect; inspect class counts and use additional validation before relying on it."
    )
else:
    error_summary = (
        f"The model made {len(errors)} errors on {len(eval_rows)} matched test rows. "
        "The next review should inspect whether errors cluster around particular positions, volumes, "
        "missing-value patterns, categories, or other lane-specific segments. Individual errors are "
        "diagnostic examples, not proof of causation."
    )

print(error_summary)


# 5) Self-check

The final checks enforce the assignment's core requirements rather than rewarding complexity.


In [ ]:
# Required checks
assert len(X_train) > 0 and len(X_test) > 0
assert len(set(idx_train) & set(idx_test)) == 0
assert len(feature_cols) > 0
assert primary_metric in comparison.columns

# No obvious label/future columns in features.
bad_feature_names = [
    c for c in feature_cols
    if any(k in norm(c) for k in [
        "future", "next_period", "post_outcome", "conversion_date",
        "label", "target", "ground_truth", "quick_win"
    ])
]
assert not bad_feature_names, f"Potential leakage features: {bad_feature_names}"

# Model must have a valid holdout result.
assert np.isfinite(model_value)
assert np.isfinite(baseline_value)

# Save a metrics receipt.
receipt = {
    "method": "Logistic Regression",
    "primary_metric": primary_metric,
    "model_primary_metric": float(model_value),
    "baseline_primary_metric": float(baseline_value),
    "delta": float(delta),
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "matched_baseline_rows": int(len(baseline_eval)),
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "target_column": TARGET_COLUMN,
    "feature_count": len(feature_cols),
    "excluded_columns": sorted(map(str, feature_exclude)),
}

receipt_path = OUTPUT_DIR / "w05_model_metrics.json"
receipt_path.write_text(json.dumps(receipt, indent=2), encoding="utf-8")

print("SELF-CHECK PASSED")
print("Metrics receipt:", receipt_path)
print(json.dumps(receipt, indent=2))


## Submission checklist

- [ ] Execute all cells with the real FlyRank dataset.
- [ ] Confirm the target column matches the Week-2/Week-3 data contract.
- [ ] Confirm no future-window or label-derived feature is included.
- [ ] Confirm the holdout split is valid and stratified.
- [ ] Confirm Week-4 baseline CSV is generated and matched to the same test rows.
- [ ] Read the model-vs-baseline table.
- [ ] Explain the error patterns rather than claiming the model is better because it is newer.
- [ ] Commit `work/notebooks/w05_model.ipynb`.
- [ ] Commit allowed JSON receipts such as `work/outputs/w05_model_metrics.json`.
- [ ] Do not commit the baseline CSV if the repository leak guard excludes it.
